In [1]:
%pip install -q pandas spacy nltk matplotlib seaborn plotly
!python -m spacy download es_core_news_md

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/42.3 MB ? eta -:--:--
     ---------------------------------------- 0.0/42.3 MB ? eta -:--:--
     ---------------------------------------- 0.0/42.3 MB ? eta -:--:--
     ---------------------------------------- 0.3/42.3 MB ? eta -:--:--
     ---------------------------------------- 0.3/42.3 MB ? eta -:--:--
      --------------------------------------- 0.8/42.3 MB 1.7 MB/s eta 0:00:25
     -- ------------------------------------- 2.4/42.3 MB 3.2 MB/s eta 0:00:13
     --- ------------------------------------ 3.9/42.3 MB 4.1 MB/s eta 0:00:10
     ----- ---------------------------------- 5.8/42.3 MB 4.9 MB/s eta 0:00:08
     ------- -------------------------------- 7.6/42.3 MB 5.6 MB/s eta 0:00:07
     --------- ------------------------------ 10.0/42.3 MB 6.3 MB/s eta 0:00:06
     ----------- ---------------------------- 12.3/42.3 MB 6.8 MB/s eta 0:00:05
     -------------- ------------------------- 14.9/42.3 MB 7.4 MB/s eta 0:00:04
     


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Cargar modelo y CSV

In [2]:
import pandas as pd
import spacy

nlp = spacy.load("es_core_news_md", disable=["ner"])

ruta = "resenas_limpias.csv"
df = pd.read_csv(ruta)

print(df.shape)
print(df.columns.tolist())
print(df["texto_es"].iloc[0][:200])

(6283, 11)
['id', 'movie_id', 'genero', 'periodo', 'anio_review', 'fecha', 'calificacion', 'valoracion', 'is_spoiler', 'texto_en', 'texto_es']
Vi esta película por primera vez cuando tenía 10 años. Tiene que ser la única película que he visto que se ha quedado conmigo, todos estos años. Ahora, 13 años después, lo he vuelto a ver, y puedo dec


In [3]:
import time

inicio = time.time()
filas = []
for id_resena, doc in zip(df["id"], nlp.pipe(df["texto_es"], batch_size=64)):
    for n_oracion, oracion in enumerate(doc.sents):
        for token in oracion:
            if token.is_space:
                continue
            filas.append({
                "id": id_resena,
                "oracion": n_oracion,
                "token": token.text,
                "lema": token.lemma_,
                "upos_spacy": token.pos_,
                "morf": str(token.morph),
            })

tokens = pd.DataFrame(filas)
tokens.to_csv("tokens_spacy.csv", index=False, encoding="utf-8")

print(tokens.shape)
print(f"{(time.time() - inicio) / 60:.1f} minutos")

(1361253, 6)
6.8 minutos


In [4]:
import nltk
from nltk.corpus import cess_esp
from nltk.tag import DefaultTagger, UnigramTagger, BigramTagger

nltk.download("cess_esp", quiet=True)

def eagles_a_upos(etiqueta):
    t = etiqueta.lower()
    if t.startswith("np"):
        return "PROPN"
    if t.startswith("cs"):
        return "SCONJ"
    if t.startswith(("va", "vs")):
        return "AUX"
    mapa = {"a": "ADJ", "c": "CCONJ", "d": "DET", "f": "PUNCT", "i": "INTJ",
            "n": "NOUN", "p": "PRON", "r": "ADV", "s": "ADP", "v": "VERB",
            "z": "NUM", "w": "NUM"}
    return mapa.get(t[:1], "X")

oraciones = [[(w.lower(), eagles_a_upos(t)) for w, t in s if w != "*0*"]
             for s in cess_esp.tagged_sents()]

corte = int(len(oraciones) * 0.9)
entreno, prueba = oraciones[:corte], oraciones[corte:]

unigrama = UnigramTagger(entreno, backoff=DefaultTagger("NOUN"))
bigrama = BigramTagger(entreno, backoff=unigrama)

print("Oraciones de entrenamiento:", len(entreno), "| prueba:", len(prueba))
print("Unigrama:", unigrama.accuracy(prueba))
print("Bigrama:", bigrama.accuracy(prueba))

[nltk_data] Error loading cess_esp: [ASN1: NOT_ENOUGH_DATA] not enough
[nltk_data]     data (_ssl.c:4057)


Oraciones de entrenamiento: 5427 | prueba: 603
Unigrama: 0.8622152746761947
Bigrama: 0.8761351793955635


In [5]:
por_oracion = tokens.groupby(["id", "oracion"], sort=False)["token"].apply(lambda s: s.str.lower().tolist())
etiquetadas = bigrama.tag_sents(por_oracion.tolist())

tokens["upos_nltk"] = [etiqueta for oracion in etiquetadas for _, etiqueta in oracion]

print(tokens.shape)
print(tokens[["token", "upos_spacy", "upos_nltk"]].head(15))

(1361253, 7)
       token upos_spacy upos_nltk
0         Vi       VERB      VERB
1       esta        DET       DET
2   película       NOUN      NOUN
3        por        ADP       ADP
4    primera        ADJ       ADJ
5        vez       NOUN      NOUN
6     cuando      SCONJ     SCONJ
7      tenía       VERB      VERB
8         10        NUM       NUM
9       años       NOUN      NOUN
10         .      PUNCT     PUNCT
11     Tiene       VERB      VERB
12       que      SCONJ     SCONJ
13       ser        AUX       AUX
14        la        DET       DET


In [6]:
tokens.to_csv("pos_analysis.csv", index=False, encoding="utf-8")

sin_puntuacion = tokens[tokens["upos_spacy"] != "PUNCT"]
coinciden = (sin_puntuacion["upos_spacy"] == sin_puntuacion["upos_nltk"]).mean()
print(f"Coinciden en el {coinciden:.1%} de los tokens (sin puntuación)")

Coinciden en el 82.3% de los tokens (sin puntuación)


In [8]:
tokens.to_csv("pos_analysis.csv.gz", index=False, compression="gzip")